In [1]:
import rasterio
from rasterio.plot import show
from rasterstats import zonal_stats
import osmnx as ox
import geopandas as gpd
import pandas as pd
import os
import numpy as np

In [2]:
field_shp = r"D:\imWEBs\STC\STC-Landuse-Data\field\unit_management_from_raster_dissolved.shp"
raster_path = r"D:\imWEBs\STC\STC-Landuse-Data\All Years\raster"
output_path = r"D:\imWEBs\STC\STC-Landuse-Data\Processed"
first_year = 1991

#load lookup tables
lookup_path = r"D:\imWEBs\STC\STC-Landuse-Data\Processed\lookups"
year_lookup = "year-lookup.csv"
crop_management_lookup = "crop-management.csv"
fer_management_lookup = "fertilizer-management.csv"
til_management_lookup = "tillage-management.csv"
graz_management_lookup = "grazing-management.csv"
field_subbasin_lookup = "field_subbasin.csv"
flag_not_proceed = "Not proceed"
col_original_crop_id = "Original ID"
col_imwebs_crop_id = "IMWEBs ID"
col_crop_code = "CropCode"
scenario = 1

In [3]:
def get_field_dominant_crop(field_shp, crop_raster):
    """
    get dominant crop for each field

    Parameters
    ----------
    field_shp : the file path of field shapefile
    crop_raster : the file path of crop raster 

    Returns
    -------
    DataFramw with two columns:
    1. Location - Field ID
    2. Original ID - Original Crop Code
    
    """
    stats = zonal_stats(field_shp, crop_raster, band=1,stats='majority count', geojson_out = True)

    field_dominont_crop = {}
    for land in stats:
        #skip if majority is none
        if land['properties']['LAND_ID'] is None or land['properties']['majority'] is None:
            continue
        else:
            field_dominont_crop[int(land['properties']['LAND_ID'])] = int(land['properties']['majority'])
    
    #create dataframe and save the original crop id to Original ID column
    df = pd.DataFrame.from_dict(field_dominont_crop, orient = 'index', columns = [col_original_crop_id])
    df.index.name = 'Location'
    df = df.sort_index()
    df = df.reset_index()

    return df

def add_year(df, current_year, first_year):
    """
    Add year columns

    Parameters
    ----------
    df: data frame
    current_year: the real year
    first_year: the first year of simulation to calcualte the year sequence
    
    """
    df['ActualYear'] = current_year
    df['Year'] = current_year - first_year + 1

    return df

def to_imwebs_crop_code(df_original_crop, year):
    """
    convert the original crop id to imwebs crop id with lookup table
    
    """

    #get name of the crop lookup csv file
    df_year_lookup = pd.read_csv(os.path.join(lookup_path, year_lookup), index_col = 0)
    crop_code_lookup = df_year_lookup.Lookup[year]

    #read crop lookup csv file and dicard not proccessed codes
    df_crop_code_lookup = pd.read_csv(os.path.join(lookup_path,crop_code_lookup + '.csv'), index_col = 0)
    df_crop_code_lookup = df_crop_code_lookup[df_crop_code_lookup[col_imwebs_crop_id]!= flag_not_proceed]
    df_crop_code_lookup[col_imwebs_crop_id] = df_crop_code_lookup[col_imwebs_crop_id].astype(int)

    #join original id and imweb code
    df_field_crop_imwebs = df_original_crop.merge(df_crop_code_lookup, how = 'inner', on = col_original_crop_id)
    df_field_crop_imwebs[col_crop_code] = df_field_crop_imwebs[col_imwebs_crop_id]

    return df_field_crop_imwebs

In [4]:
#for crop management
years = np.arange(first_year, 2021, 1)

field_crops = []
for y in years:   
    print(f"year: {y}")
    crop_raster = os.path.join(raster_path, f'{y}.tif')

    df = get_field_dominant_crop(field_shp, crop_raster)
    df = add_year(df, y, first_year)
    df = to_imwebs_crop_code(df, y)
    
    field_crops.append(df)
    
df_field_crop = pd.concat(field_crops,ignore_index=True)

df_field_crop.to_csv(os.path.join(output_path, 'dominant_crop.csv'))

#add scenario column
df_field_crop['Scenario'] = scenario


year: 1991


KeyboardInterrupt: 

In [ ]:
df_field_crop.to_csv(os.path.join(output_path, 'dominant_crop.csv'))

In [ ]:
#create random date between first and last day
import random

col_planting_date = ('PlantingMon', 'PlantingDay')
col_harvest_date = ('HarvestMon', 'HarvestDay')
col_fertilizer_date = ('FerMon', 'FerDay')
col_tillage_date = ('TillMon', 'TillDay')
value_use_planting_date = -1
value_30_after_harvest = -30


#create random date with given first month, day and last month and day
def create_random_date(mon_first, day_first, mon_last, day_last):
    #create random planting month and day
    first_day = pd.Timestamp(year = 2000, month = mon_first, day = day_first)
    last_day = pd.Timestamp(year = 2000, month = mon_last, day = day_last)
    day_range = last_day - first_day
    ran = random.randint(0, day_range.days)
    ran_day = first_day + pd.Timedelta(f'{ran} days')
    return (ran_day.month, ran_day.day)

def offset_days(mon, day, offset_days):
    d = pd.Timestamp(year = 2000, month = mon, day = day)
    d = d + pd.Timedelta(f'{offset_days} days')
    
    return (d.month, d.day)

#populate the random date for given index and mon/day column
def populate_random_date(df, index, col_mon, col_day):
    mon_first = df.loc[index, f'{col_mon}_First']
    day_first = df.loc[index, f'{col_day}_First']
    mon_last = df.loc[index, f'{col_mon}_Last']
    day_last = df.loc[index, f'{col_day}_Last']
       
    
    #if one of the mon/day is -1, then planting month/day will be used
    if mon_first == value_use_planting_date | day_first == value_use_planting_date | mon_last == value_use_planting_date | day_last == value_use_planting_date: 
        df.loc[index, col_mon] = df.loc[index, col_planting_date[0]]
        df.loc[index, col_day] = df.loc[index, col_planting_date[1]]
    #if one of the mon/day is -30, then use 30 days after harvest date
    elif mon_first == value_30_after_harvest | day_first == value_30_after_harvest| mon_last == value_30_after_harvest | day_last == value_30_after_harvest: 
        offset = offset_days(df.loc[index, col_harvest_date[0]], df.loc[index, col_harvest_date[1]], 30)
        df.loc[index, col_mon] = offset[0]
        df.loc[index, col_day] = offset[1]        
    else:    
        ran = create_random_date(mon_first, day_first, mon_last, day_last)    
        df.loc[index, col_mon] = ran[0]
        df.loc[index, col_day] = ran[1]

#make sure the planting day of winter wheat is after the harvest day of previous crop
#if not, set it to the harvest day of previous crop
def move_winter_wheat_planting_day(df, index):
    #only apply to crop where the previous year flag is 1
    if df.loc[index, 'Pervious Year'] != 1:
        return
    #only apply to crop on the second years and after as the first year will be set to Jan 1st
    year = df.loc[index, 'Year']
    if year == 1:
        return
    #get crop of previous year
    location = df.loc[index, 'Location']
    previous_crop_df = df[(df['Year'] == year - 1) & (df['Location'] == location)]

    #return if previous crop doesn't exist
    if len(previous_crop_df.index) == 0:
        return

    #get harvest date
    harvest_mon = previous_crop_df.loc[previous_crop_df.index[0],col_harvest_date[0]]
    harvest_day = previous_crop_df.loc[previous_crop_df.index[0],col_harvest_date[1]]
    harvest_date = pd.Timestamp(year = 2000, month = harvest_mon, day = harvest_day)

    #get planting date
    planting_mon = df.loc[index,col_planting_date[0]]
    planting_day = df.loc[index,col_planting_date[1]]
    planting_date = pd.Timestamp(year = 2000, month = planting_mon, day = planting_day)

    #print(f"Location = {location}, Year = {year}, Planting = {planting_mon}-{planting_day}, Previous Harvest = {harvest_mon}-{harvest_day}")

    #check planting date
    if planting_date >= harvest_date:
        return

    #chang it to harvest date
    df.loc[index,col_planting_date[0]] = harvest_mon
    df.loc[index,col_planting_date[1]] = harvest_day

    #print(f"Changed, Planting = {df.loc[index,col_planting_date[0]]}-{df.loc[index,col_planting_date[1]]}")

#-------------------------------------------------------------------------------------------------
#crop management
df_crop_management_lookup = pd.read_csv(os.path.join(lookup_path, crop_management_lookup))
df_crop = df_field_crop.merge(df_crop_management_lookup, how = 'inner', on = col_crop_code)

#add planting and harvest month and day column
for col in col_planting_date + col_harvest_date:
    df_crop[col] = -1
    
#add ID column
df_crop['ID'] = df_crop['Location']

#populate the columns
for index in df_crop.index:    
    #planting dates
    populate_random_date(df_crop, index, col_planting_date[0], col_planting_date[1])
    
    #harvest dates
    populate_random_date(df_crop, index, col_harvest_date[0], col_harvest_date[1])

for index in df_crop.index: 
    #change the planting date of crops that are planted in previous year
    move_winter_wheat_planting_day(df_crop, index)

#Get the year. If previous year flag is 1, then previous year is used.
df_crop['Year'] = np.where(df_crop['Pervious Year'] == 1, df_crop['Year'] - 1, df_crop['Year'])

#if the year is zero, then it's winter wheat and we will use year 1 and set plant day to Jan 1st
df_crop[col_planting_date[0]] = np.where(df_crop['Year'] == 0, 1, df_crop[col_planting_date[0]])
df_crop[col_planting_date[1]] = np.where(df_crop['Year'] == 0, 1, df_crop[col_planting_date[1]])
df_crop['Year'] = np.where(df_crop['Year'] == 0, 1, df_crop['Year'])
    
#save crop managment
df_crop.to_csv(os.path.join(output_path, 'crop_management.csv'), index = False, columns = ['Location','Scenario','ID','ActualYear','CropCode',
                                                                                           'Year','PlantingDay','PlantingMon','HarvestDay','HarvestMon',
                                                                                           'HarvestEfficiency','HarvestType','HarvestIndexOverride','CNOP',
                                                                                           'StoverFraction','IsGrain','PRCOP'])

#-------------------------------------------------------------------------------------------------
#fertilizer management
df_fer_management_lookup = pd.read_csv(os.path.join(lookup_path, fer_management_lookup))
df_fertilizer = df_crop.merge(df_fer_management_lookup, how = 'inner', on = col_crop_code)

#add the columns
for col in col_fertilizer_date:
    df_fertilizer[col] = -1

#populate the columns
for index in df_fertilizer.index:    
    #fertilizer dates
    populate_random_date(df_fertilizer, index, col_fertilizer_date[0], col_fertilizer_date[1])
    
#save fertilizer managment
df_fertilizer.to_csv(os.path.join(output_path, 'fertilizer_management.csv'), index = False, columns = ['Scenario','Location','Year','FerMon','FerDay',
                                                                                                       'FerType','FerRate','FerSurface'])
    
    
#-------------------------------------------------------------------------------------------------
#tillage management   
df_til_management_lookup = pd.read_csv(os.path.join(lookup_path, til_management_lookup))
df_tillage = df_crop.merge(df_til_management_lookup, how = 'inner', on = col_crop_code)
    
#add the columns
for col in col_tillage_date:
    df_tillage[col] = -1
        
#populate the columns
for index in df_tillage.index:
    #tillage dates
    populate_random_date(df_tillage, index, col_tillage_date[0], col_tillage_date[1])
    
#save tillage managment
df_tillage.to_csv(os.path.join(output_path, 'tillage_management.csv'), index = False, columns = ['Scenario','Location','Year','TillMon','TillDay','TillCode'])

In [ ]:
#grazing management
df_grazaing_management_lookup = pd.read_csv(os.path.join(lookup_path, graz_management_lookup))
df_grazing = df_field_crop.merge(df_grazaing_management_lookup, how = 'inner', on = col_crop_code)

#get field to subbasin mapping
df_field_subbasin_lookup = pd.read_csv(os.path.join(lookup_path, field_subbasin_lookup))
df_field_subbasin_lookup = df_field_subbasin_lookup.rename(columns={"FIELD": "Location", "SUBBASIN": "SourceID"})
df_grazing = df_grazing.merge(df_field_subbasin_lookup, how = 'inner', on = 'Location')

#save grazing managment
df_grazing.to_csv(os.path.join(output_path, 'grazing_management.csv'), index = False, columns = ['Scenario','Location','Year','GraMon','GraDay','Days',
'Ani_ID','Ani_adult','GR_Density','DayFra','Source', 'SourceID','Dugout_ID','Access','Fencing','StreamAniPerc', 'Drinking_time','BankK_Change'])